In [2]:
import pandas as pd
import numpy as np

# Load the file
file_name = 'ARC_GOLD_MASTER_BT.csv'
df = pd.read_csv(file_name)

# Cleaning
df.columns = [c.strip() for c in df.columns]
df['TIME'] = pd.to_datetime(df['TIME'])
df = df.sort_values('TIME')

# Detect price jumps between consecutive event logs
df['price_change'] = df['BID'].diff()

# Define a 'Major Jump' (potential trend starter)
# In Boom 1000, a significant jump in this low-resolution log might be > 10 points
jump_threshold = 10.0
jumps = df[df['price_change'] >= jump_threshold].copy()

trends = []

for idx in jumps.index:
    start_time = df.loc[idx, 'TIME']
    start_bid = df.loc[idx, 'BID']
    entry_bid = df.loc[idx-1, 'BID'] if idx > 0 else start_bid

    # Look ahead to see how long it stays up
    look_ahead = df.loc[idx+1:]
    end_time = start_time
    max_bid = start_bid

    for i, row in look_ahead.iterrows():
        # If price drops below the pre-jump level, the 'trend' or 'burst' ended
        if row['BID'] < entry_bid:
            break
        end_time = row['TIME']
        if row['BID'] > max_bid:
            max_bid = row['BID']

        # Limit look ahead to 5 hours to avoid infinite loops or merged trends
        if (row['TIME'] - start_time).total_seconds() > 5 * 3600:
            break

    duration_mins = (end_time - start_time).total_seconds() / 60

    if duration_mins >= 30: # Only consider movements longer than 30 mins
        trends.append({
            'start': start_time,
            'end': end_time,
            'duration_min': duration_mins,
            'pips_total': max_bid - entry_bid,
            'start_node': df.loc[idx, 'H6_FRAC'],
            'hour': start_time.hour
        })

trends_df = pd.DataFrame(trends)

if not trends_df.empty:
    print("Trend Analysis Summary:")
    print(f"Total Trends found (>30min): {len(trends_df)}")
    print("\nDuration Statistics (min):")
    print(trends_df['duration_min'].describe())

    print("\nTop Hours for Trends (UTC/Server Time):")
    print(trends_df['hour'].value_counts().sort_index())

    print("\nTop H6 Nodes where Trends Start:")
    print(trends_df['start_node'].value_counts().head(10))

    # Identify 'Grand Colossus' trends (> 120 mins)
    big_trends = trends_df[trends_df['duration_min'] >= 120]
    print(f"\nTotal 'Grand Colossus' Trends (>120min): {len(big_trends)}")
    if not big_trends.empty:
        print("Example Big Trends:")
        print(big_trends[['start', 'duration_min', 'pips_total', 'start_node']].head(10))
else:
    print("No sustained trends (>30min) found with the current logic.")

FileNotFoundError: [Errno 2] No such file or directory: 'ARC_GOLD_MASTER_BT.csv'

In [3]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving ARC_GOLD_MASTER_BT.csv to ARC_GOLD_MASTER_BT.csv
User uploaded file "ARC_GOLD_MASTER_BT.csv" with length 9393616 bytes


In [4]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(trends_df['duration_min'], bins=20, kde=True)
plt.title('Distribution of Trend Durations (Minutes)')
plt.xlabel('Duration (Minutes)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

NameError: name 'trends_df' is not defined

<Figure size 1000x600 with 0 Axes>

After uploading, the file will be available in the Colab environment. You can then re-run the previous code cell to process it.